In [2]:
from pathlib import Path

# Comme le notebook est dans notebooks/, on remonte d'un niveau
DATA_DIR = Path("../data")

for item in DATA_DIR.iterdir():
    print(item)

..\data\ann_txt_files
..\data\DetectOnco_Final.csv
..\data\GuideAnnotation.docx


structure du csv

In [3]:
import pandas as pd

df = pd.read_csv(DATA_DIR / "DetectOnco_Final.csv")

print("Colonnes :", df.columns.tolist())
print("Nombre de lignes :", len(df))
print("\nTypes d'entités (label) :")
print(df['label'].unique())
print("\nNombre de documents uniques :")
print(df['doc_name'].nunique())

Colonnes : ['doc_name', 'id', 'label', 'code', 'content', 'full_span']
Nombre de lignes : 71126

Types d'entités (label) :
<StringArray>
['topographie', 'differenciation', 'morphologie', 'expression_CIM']
Length: 4, dtype: str

Nombre de documents uniques :
1300


structure du fichier ann

In [4]:
ann_dir = DATA_DIR / "ann_txt_files"
ann_files = sorted(ann_dir.glob("*.ann"))
print(f"Nombre de fichiers .ann : {len(ann_files)}")

# Juste la structure d'un fichier, ligne par ligne, avec repr() pour voir le format exact
with open(ann_files[0], "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"\nNombre de lignes dans {ann_files[0].name} :", len(lines))
print("Format de la première ligne (repr) :")
print(repr(lines[0]) if lines else "fichier vide")

Nombre de fichiers .ann : 1301

Nombre de lignes dans cc_onco1.ann : 40
Format de la première ligne (repr) :
'T1\tmorphologie 2300 2311\tadénopathie\n'


distribution des longueurs de texte

In [5]:
txt_files = sorted(ann_dir.glob("*.txt"))
print(f"Nombre de fichiers .txt : {len(txt_files)}")

lengths = []
for f in txt_files:
    content = f.read_text(encoding="utf-8")
    lengths.append(len(content))

import statistics
print(f"Longueur moyenne : {statistics.mean(lengths):.0f} caractères")
print(f"Min : {min(lengths)}, Max : {max(lengths)}")

Nombre de fichiers .txt : 1301
Longueur moyenne : 5287 caractères
Min : 1300, Max : 16321


In [6]:
with open(ann_files[0], "r", encoding="utf-8") as f:
    lines = f.readlines()

# Voir les types de préfixes de ligne (T, #, N, R...)
prefixes = [line.split('\t')[0][0] for line in lines if line.strip()]
from collections import Counter
print(Counter(prefixes))

Counter({'T': 20, '#': 20})


In [7]:
with open(ann_files[0], "r", encoding="utf-8") as f:
    lines = f.readlines()

# Isoler une ligne T et la ligne # qui lui correspond probablement
t_lines = [l for l in lines if l.startswith('T')]
hash_lines = [l for l in lines if l.startswith('#')]

print("Exemple ligne T :")
print(repr(t_lines[0]))

print("\nExemple ligne # :")
print(repr(hash_lines[0]))

Exemple ligne T :
'T1\tmorphologie 2300 2311\tadénopathie\n'

Exemple ligne # :
'#1\tICD-O T1\t8000/1\n'


détection de spans discontinus 

In [8]:
from pathlib import Path

ann_files = sorted((DATA_DIR / "ann_txt_files").glob("*.ann"))
discontinuous_count = 0

for ann_path in ann_files:
    with open(ann_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("T") and ";" in line.split("\t")[1]:
                discontinuous_count += 1
                print(f"Span discontinu trouvé : {ann_path.name}")
                break

print(f"\nTotal fichiers avec spans discontinus : {discontinuous_count}")

Span discontinu trouvé : cc_onco1.ann
Span discontinu trouvé : cc_onco1000.ann
Span discontinu trouvé : cc_onco1006.ann
Span discontinu trouvé : cc_onco1008.ann
Span discontinu trouvé : cc_onco1009.ann
Span discontinu trouvé : cc_onco1010.ann
Span discontinu trouvé : cc_onco1011.ann
Span discontinu trouvé : cc_onco1016.ann
Span discontinu trouvé : cc_onco1019.ann
Span discontinu trouvé : cc_onco102.ann
Span discontinu trouvé : cc_onco1020.ann
Span discontinu trouvé : cc_onco1021.ann
Span discontinu trouvé : cc_onco1022.ann
Span discontinu trouvé : cc_onco1023.ann
Span discontinu trouvé : cc_onco1025.ann
Span discontinu trouvé : cc_onco1027.ann
Span discontinu trouvé : cc_onco103.ann
Span discontinu trouvé : cc_onco1031.ann
Span discontinu trouvé : cc_onco1038.ann
Span discontinu trouvé : cc_onco104.ann
Span discontinu trouvé : cc_onco1041.ann
Span discontinu trouvé : cc_onco1043.ann
Span discontinu trouvé : cc_onco1046.ann
Span discontinu trouvé : cc_onco1049.ann
Span discontinu trouvé

In [9]:
for ann_path in ann_files:
    with open(ann_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("T") and ";" in line.split("\t")[1]:
                print(ann_path.name)
                print(repr(line))
                break
        else:
            continue
        break

cc_onco1.ann
'T13\tmorphologie 3225 3234;3246 3258\tcarcinome microcytaire\n'


In [10]:
import importlib
from onco_ner.data import brat_parser
from onco_ner import schemas
importlib.reload(schemas)
importlib.reload(brat_parser)

doc = brat_parser.parse_document(ann_dir / "cc_onco1.txt", ann_dir / "cc_onco1.ann")
discontinuous = [e for e in doc["entities"] if e.is_discontinuous]
print(f"Entités discontinues dans cc_onco1 : {len(discontinuous)}")
print(discontinuous[0] if discontinuous else "aucune")

# Test d'immuabilité
try:
    discontinuous[0].icdo_code = "TEST"
    print("ERREUR : l'objet n'est pas gelé !")
except Exception as e:
    print(f"OK, objet bien immuable : {type(e).__name__}")

# Test de hachabilité
print(f"Hachable : {hash(discontinuous[0]) is not None}")

22:19:09 [INFO] onco_ner.data.brat_parser: Span discontinu : cc_onco1.ann (T13), 2 sous-spans
22:19:09 [INFO] onco_ner.data.brat_parser: cc_onco1.ann : 20 entités parsées
Entités discontinues dans cc_onco1 : 1
id='T13' label='morphologie' spans=(Span(start=3225, end=3234), Span(start=3246, end=3258)) text='carcinome microcytaire' icdo_code='8041/3'
OK, objet bien immuable : ValidationError
Hachable : True


In [12]:
from onco_ner.data.brat_parser import parse_corpus

ann_dir = DATA_DIR / "ann_txt_files"
documents = parse_corpus(ann_dir)

total_entities = sum(len(doc["entities"]) for doc in documents)
discontinuous_docs = sum(
    1 for doc in documents
    if any(e.is_discontinuous for e in doc["entities"])
)

print(f"Documents parsés : {len(documents)}")
print(f"Total entités : {total_entities}")
print(f"Total attendu (CSV) : 71126")
print(f"Écart : {total_entities - 71126}")
print(f"Documents avec spans discontinus : {discontinuous_docs}")

00:26:24 [INFO] onco_ner.data.brat_parser: Span discontinu : cc_onco1.ann (T13), 2 sous-spans
00:26:24 [INFO] onco_ner.data.brat_parser: cc_onco1.ann : 20 entités parsées
00:26:24 [INFO] onco_ner.data.brat_parser: cc_onco10.ann : 107 entités parsées
00:26:24 [INFO] onco_ner.data.brat_parser: cc_onco100.ann : 67 entités parsées
00:26:24 [INFO] onco_ner.data.brat_parser: Span discontinu : cc_onco1000.ann (T27), 2 sous-spans
00:26:24 [INFO] onco_ner.data.brat_parser: cc_onco1000.ann : 52 entités parsées
00:26:25 [INFO] onco_ner.data.brat_parser: cc_onco1001.ann : 19 entités parsées
00:26:25 [INFO] onco_ner.data.brat_parser: Span discontinu : cc_onco1006.ann (T6), 2 sous-spans
00:26:25 [INFO] onco_ner.data.brat_parser: Span discontinu : cc_onco1006.ann (T17), 2 sous-spans
00:26:25 [INFO] onco_ner.data.brat_parser: cc_onco1006.ann : 79 entités parsées
00:26:25 [INFO] onco_ner.data.brat_parser: cc_onco1007.ann : 23 entités parsées
00:26:25 [INFO] onco_ner.data.brat_parser: Span discontinu : 

In [1]:
import pandas as pd
df = pd.read_csv("../data/DetectOnco_Final.csv")
print(df['label'].value_counts())
print("\nExemple full_span :")
print(df['full_span'].head(3).tolist())

label
morphologie        25634
expression_CIM     25634
topographie        18805
differenciation     1053
Name: count, dtype: int64

Exemple full_span :
['3899 3924', '2136 2149', '3051 3065']


test rapide

In [3]:
import sys
sys.path.insert(0, "../src")

import importlib
from onco_ner.data import preprocessing
importlib.reload(preprocessing)

from onco_ner.data.preprocessing import load_corpus_csv, split_corpus, save_splits
from pathlib import Path

DATA_DIR = Path("../data")

# Charger
df = load_corpus_csv(DATA_DIR / "DetectOnco_Final.csv")
print(df.head(3))
print(df.dtypes)

# Splitter
train_df, val_df, test_df = split_corpus(df, seed=42)

# Vérifier la distribution de differenciation dans chaque split
for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    diff_count = split.filter(pl.col("label") == "differenciation").shape[0]
    print(f"{name} — differenciation : {diff_count} entités")

# Sauvegarder
save_splits(train_df, val_df, test_df, DATA_DIR / "splits")
print("Splits sauvegardés !")

13:24:06 [INFO] onco_ner.data.preprocessing: Corpus chargé : 71126 entités, 1300 documents
13:24:06 [INFO] onco_ner.data.preprocessing: Distribution des labels :
shape: (4, 2)
┌─────────────────┬───────┐
│ label           ┆ count │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞═════════════════╪═══════╡
│ differenciation ┆ 1053  │
│ expression_CIM  ┆ 25634 │
│ morphologie     ┆ 25634 │
│ topographie     ┆ 18805 │
└─────────────────┴───────┘
shape: (3, 7)
┌─────────────────┬─────┬─────────────┬───────┬───────────────────────────┬────────────┬──────────┐
│ doc_name        ┆ id  ┆ label       ┆ code  ┆ content                   ┆ span_start ┆ span_end │
│ ---             ┆ --- ┆ ---         ┆ ---   ┆ ---                       ┆ ---        ┆ ---      │
│ str             ┆ str ┆ str         ┆ str   ┆ str                       ┆ i32        ┆ i32      │
╞═════════════════╪═════╪═════════════╪═══════╪═══════════════════════════╪════════════╪══════════╡
│ cc_onco982.ann  ┆ T44 ┆ topo

In [1]:
import polars as pl
print(pl.__version__)

1.9.0


In [7]:
import sys
print(sys.executable)

C:\Users\jeane\Documents\Alternance\aphp\FRACCO\onco-ner\.venv-onco\Scripts\python.exe
